# Shelf to Tales - Spoiler Detection Model Training

This notebook trains per-book spoiler detection models using **unsloth** + **LoRA fine-tuning**.

## Setup
1. Enable GPU: Runtime → Change runtime type → T4 GPU
2. Set your `WEBHOOK_URL` in the configuration cell below
3. Run all cells: Runtime → Run all
4. Upload training JSONL files to Google Drive folder: `shelftotales-training/`

## Step 1: Configuration

Set your backend webhook URL below. This is where Colab will notify your backend when training completes.

- **Local backend**: Use ngrok URL (see instructions below)
- **Deployed backend**: Use your production URL
- **Skip webhook**: Leave `WEBHOOK_URL` empty

In [ ]:
# ============ CONFIGURATION - EDIT THIS ============

# Your backend webhook URL
# For local: use ngrok (run: ngrok http 8080)
# For deployed: use your production URL
WEBHOOK_URL = ""  # e.g., "https://abc123.ngrok.io/api/ai/webhooks/training-complete"

# Google Drive folder name (will be created automatically)
DRIVE_FOLDER = "shelftotales-training"

# Base model for fine-tuning
BASE_MODEL = "unsloth/llama-3-8b-Instruct"

# How often to check for new training data (seconds)
POLL_INTERVAL = 60

# ===============================================

print(f"Configuration set:")
print(f"  Webhook URL: {WEBHOOK_URL or '(not set - manual mode)'}")
print(f"  Drive Folder: {DRIVE_FOLDER}")
print(f"  Base Model: {BASE_MODEL}")
print(f"  Poll Interval: {POLL_INTERVAL}s")

## Step 2: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
training_dir = Path(f"/content/drive/MyDrive/{DRIVE_FOLDER}")
training_dir.mkdir(parents=True, exist_ok=True)
print(f"Google Drive mounted!")
print(f"Training data folder: {training_dir}")
print(f"\nUpload your JSONL files to: {training_dir}")

## Step 3: Install Dependencies

In [ ]:
import subprocess
import sys

def install_package(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("Installing unsloth and dependencies...")
install_package("unsloth")
install_package("transformers")
install_package("datasets")
install_package("trl")
install_package("accelerate")
install_package("bitsandbytes")

print("\n✓ All dependencies installed!")

# Verify GPU
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\n✓ GPU detected: {gpu_name} ({gpu_memory:.1f} GB)")
else:
    print("\n✗ WARNING: No GPU detected! Training will be very slow.")
    print("  Go to Runtime → Change runtime type → T4 GPU")

## Step 4: Define Training Functions

In [ ]:
import json
import time
import requests
import shutil
from pathlib import Path

def check_for_new_training_data():
    """Check Drive folder for new training JSONL files."""
    jsonl_files = list(training_dir.glob("spoiler-train-*.jsonl"))
    trained_marker = training_dir / ".trained"

    trained_books = set()
    if trained_marker.exists():
        with open(trained_marker) as f:
            trained_books = set(f.read().splitlines())

    new_files = []
    for f in jsonl_files:
        book_id = f.stem.replace("spoiler-train-", "")
        if book_id not in trained_books:
            new_files.append((book_id, f))

    return new_files

def train_model(book_id, training_data_path):
    """Train a model for a specific book using unsloth."""
    print(f"\n{'='*60}")
    print(f"Training model for book ID: {book_id}")
    print(f"Training data: {training_data_path}")
    print(f"{'='*60}\n")

    from unsloth import FastLanguageModel
    from trl import SFTTrainer
    from transformers import TrainingArguments
    from datasets import load_dataset

    # Load base model
    print("Loading base model...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL,
        max_seq_length=2048,
        dtype=None,
        load_in_4bit=True,
    )

    # Add LoRA adapters
    print("Adding LoRA adapters...")
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
    )

    # Load training data
    print("Loading training data...")
    dataset = load_dataset('json', data_files=str(training_data_path), split='train')
    print(f"Loaded {len(dataset)} training examples")

    # Training config
    training_args = TrainingArguments(
        output_dir=f"./results/book-{book_id}",
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        save_strategy="epoch",
        optim="adamw_8bit",
    )

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset,
        dataset_text_field="messages",
        max_seq_length=2048,
        args=training_args,
    )

    # Train
    print("\nStarting training...")
    trainer.train()

    # Save LoRA adapter
    lora_path = f"./models/spoiler-{book_id}-lora"
    model.save_pretrained(lora_path)
    print(f"\nLoRA adapter saved to: {lora_path}")

    # Export to GGUF
    print("Exporting to GGUF (q4_k_m)...")
    gguf_path = f"./models/spoiler-{book_id}-gguf"
    model.save_pretrained_gguf(gguf_path, tokenizer, quantization_method="q4_k_m")
    print(f"GGUF model saved to: {gguf_path}/")

    return gguf_path

print("✓ Training functions defined!")

## Step 5: Define Helper Functions

In [ ]:
def create_ollama_model(book_id, gguf_path):
    """Create Ollama model files from GGUF."""
    model_name = f"shelf-spoiler-book-{book_id}"
    gguf_file = list(Path(gguf_path).glob("*.gguf"))[0]

    modelfile_content = f"""FROM {gguf_file}

TEMPLATE """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a spoiler detection model. Classify book reviews as:
- SAFE: No spoilers
- MINOR_SPOILER: Hints about plot or characters
- MAJOR_SPOILER: Reveals key plot points, endings, or character deaths

You MUST respond with ONLY valid JSON:
{{
  "level": "SAFE" | "MINOR_SPOILER" | "MAJOR_SPOILER",
  "score": 0.0-1.0,
  "reasoning": "brief explanation"
}}<|eot_id|><|start_header_id|>user<|end_header_id|>
Analyze this review for spoilers:
{{{{ .Prompt }}}}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""

PARAMETER temperature 0.1
PARAMETER num_predict 200
PARAMETER top_p 0.9
PARAMETER repeat_penalty 1.1
"""

    modelfile_path = Path(gguf_path) / "Modelfile"
    with open(modelfile_path, 'w') as f:
        f.write(modelfile_content)

    print(f"Created Modelfile for {model_name}")
    return model_name, modelfile_path

def notify_backend(book_id, model_name, status, gguf_drive_file_id=None):
    """Send webhook notification to backend."""
    if not WEBHOOK_URL:
        print("  No webhook URL configured. Manual notification required.")
        print(f"  Run this locally: ollama create {model_name} -f /path/to/Modelfile")
        return

    payload = {
        "bookId": int(book_id),
        "modelName": model_name,
        "status": status,
    }
    if gguf_drive_file_id:
        payload["ggufDriveFileId"] = gguf_drive_file_id

    try:
        response = requests.post(WEBHOOK_URL, json=payload, timeout=10)
        print(f"  Backend notified: {response.status_code}")
    except Exception as e:
        print(f"  Failed to notify backend: {e}")

def upload_to_drive(local_path, drive_path):
    """Upload a file to Google Drive."""
    src = Path(local_path)
    dst = training_dir / drive_path
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    print(f"  Uploaded to: {dst}")
    return str(dst)

def mark_as_trained(book_id):
    """Mark a book as trained to avoid re-training."""
    trained_marker = training_dir / ".trained"
    with open(trained_marker, 'a') as f:
        f.write(f"{book_id}\n")

print("✓ Helper functions defined!")

## Step 6: Check for Training Data

Upload your JSONL files to the `shelftotales-training` folder in Google Drive, then run this cell to check.

In [ ]:
new_books = check_for_new_training_data()

if new_books:
    print(f"Found {len(new_books)} book(s) ready for training:")
    for book_id, path in new_books:
        print(f"  - Book {book_id}: {path.name}")
else:
    print("No new training data found.")
    print(f"\nTo train a model:")
    print(f"1. Generate JSONL: POST http://localhost:8080/api/ai/webhooks/books/{{bookId}}/check-training")
    print(f"2. Upload the file to: {training_dir}")
    print(f"3. Re-run this cell")

## Step 7: Train All Models

Run this to start the training loop. It will:
1. Check for new JSONL files every 60 seconds
2. Train a model for each new book
3. Export to GGUF format
4. Notify your backend (if webhook is set)

**Press the stop button** when you want to stop the loop.

In [ ]:
print("Starting training loop...")
print(f"Checking for new data every {POLL_INTERVAL} seconds")
print("Press the STOP button to stop the loop\n")

try:
    while True:
        new_books = check_for_new_training_data()

        if new_books:
            print(f"\n{'='*60}")
            print(f"Found {len(new_books)} book(s) to train")
            print(f"{'='*60}")

            for book_id, training_path in new_books:
                try:
                    # Train the model
                    gguf_path = train_model(book_id, training_path)

                    # Create Ollama model files
                    model_name, modelfile_path = create_ollama_model(book_id, gguf_path)

                    # Upload GGUF to Drive
                    gguf_file = list(Path(gguf_path).glob("*.gguf"))[0]
                    drive_gguf_path = upload_to_drive(gguf_file, f"models/{book_id}/{gguf_file.name}")

                    # Upload Modelfile to Drive
                    upload_to_drive(modelfile_path, f"models/{book_id}/Modelfile")

                    # Notify backend
                    notify_backend(book_id, model_name, "COMPLETE")

                    # Mark as trained
                    mark_as_trained(book_id)

                    print(f"\n✓ Training complete for book {book_id}")
                    print(f"  Model: {model_name}")
                    print(f"  GGUF: {drive_gguf_path}")

                except Exception as e:
                    print(f"\n✗ Training failed for book {book_id}: {e}")
                    notify_backend(book_id, f"shelf-spoiler-book-{book_id}", "FAILED")
        else:
            print(f"  No new data. Checking again in {POLL_INTERVAL}s...", end="\r")

        time.sleep(POLL_INTERVAL)

except KeyboardInterrupt:
    print("\n\nTraining loop stopped.")

## Step 8: Train a Single Book (Optional)

Use this to train a specific book manually instead of using the loop.

In [ ]:
# Set the book ID and path to the JSONL file
BOOK_ID = "123"  # Change this to your book ID
JSONL_FILE = f"/content/drive/MyDrive/{DRIVE_FOLDER}/spoiler-train-{BOOK_ID}.jsonl"

# Check if file exists
jsonl_path = Path(JSONL_FILE)
if not jsonl_path.exists():
    print(f"File not found: {JSONL_FILE}")
    print(f"Upload your JSONL to: {training_dir}")
else:
    print(f"Training book {BOOK_ID}...")
    gguf_path = train_model(BOOK_ID, jsonl_path)
    model_name, modelfile_path = create_ollama_model(BOOK_ID, gguf_path)

    # Upload to Drive
    gguf_file = list(Path(gguf_path).glob("*.gguf"))[0]
    upload_to_drive(gguf_file, f"models/{BOOK_ID}/{gguf_file.name}")
    upload_to_drive(modelfile_path, f"models/{BOOK_ID}/Modelfile")

    # Notify backend
    notify_backend(BOOK_ID, model_name, "COMPLETE")
    mark_as_trained(BOOK_ID)

    print(f"\n✓ Done! Model: {model_name}")

## Step 9: Download Model (After Training)

After training completes, download the GGUF file from Google Drive and create the Ollama model locally.

In [ ]:
# List all trained models
models_dir = training_dir / "models"
if models_dir.exists():
    print("Trained models in Google Drive:\n")
    for model_dir in sorted(models_dir.iterdir()):
        if model_dir.is_dir():
            gguf_files = list(model_dir.glob("*.gguf"))
            modelfile = model_dir / "Modelfile"
            print(f"Book {model_dir.name}:")
            for f in gguf_files:
                print(f"  GGUF: {f}")
            if modelfile.exists():
                print(f"  Modelfile: {modelfile}")
            print()
else:
    print("No trained models yet.")
    print("Run the training loop first.")

## Setup Instructions

### For Local Backend (Development)

1. Start your backend:
```bash
cd backend/shelfToTales
./mvnw spring-boot:run
```

2. Install and run ngrok:
```bash
brew install ngrok
ngrok http 8080
```

3. Copy the ngrok URL and set it in Cell 1:
```python
WEBHOOK_URL = "https://abc123.ngrok.io/api/ai/webhooks/training-complete"
```

### For Deployed Backend

Set your production URL in Cell 1:
```python
WEBHOOK_URL = "https://your-app.vercel.app/api/ai/webhooks/training-complete"
```

### Manual Mode (No Webhook)

Leave `WEBHOOK_URL` empty. After training:
1. Download GGUF from Google Drive
2. Run locally: `ollama create shelf-spoiler-book-123 -f Modelfile`
3. Notify backend:
```bash
curl -X POST http://localhost:8080/api/ai/webhooks/training-complete \
  -H "Content-Type: application/json" \
  -d '{"bookId": 123, "modelName": "shelf-spoiler-book-123", "status": "COMPLETE"}'
```